# CrewAI Demo

CrewAI orchestrates role-based multi-agent teams that mimic real-world human crew dynamics.
The flow is:

LLM -> Agents -> tasks -> Crew -> Process -> kickoff -> crew output

## Required Environment Variables

```
OPENAI_API_KEY
SERPER_API_KEY  # Optional, for web search
```
## Sign up for free SERPER_API_KEY
https://serper.dev/

In [ ]:
# If running standalone without the project setup:
# pip install crewai crewai-tools openai python-dotenv

In [1]:
# Setup
%pip install -q \
    "crewai==1.15.18" \
    "crewai-tools==1.15.18" \
    openai \
    python-dotenv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 836.4/836.4 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.2/197.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

1) Load environment variables - Please read instructions carefully

In [2]:
# if you are running in google colab, getpass() to securely enter the API key without displaying it.
import os
from getpass import getpass


# Ensure the OPEN AI API key is always prompted by clearing it if already set
if "OPENAI_API_KEY" in os.environ:
    del os.environ["OPENAI_API_KEY"]

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

  # Ensure SERPER_API_KEY key is always prompted by clearing it if already set
if "SERPER_API_KEY" in os.environ:
    del os.environ["SERPER_API_KEY"]

if not os.getenv("SERPER_API_KEY"):
    os.environ["SERPER_API_KEY"] = getpass("Enter your serper API key: ")

Enter your OpenAI API key: ··········
Enter your serper API key: ··········


In [3]:
import os
import yaml
from pathlib import Path
from dotenv import load_dotenv
from typing import Any, Type, Optional
from pydantic import BaseModel, Field

# Load environment variables
#load_dotenv(override=True)

# Import CrewAI components
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool
from crewai.tools import BaseTool

## What We're Building

A **Content Marketing Team** — three agents (researcher, writer, editor) collaborating to produce a blog post about AI-powered content marketing.

In [4]:
llm = LLM(
    model="gpt-4o",
    temperature=0.7,
    store=True # Store the LLM responses in the Crew memory for future reference
)

web_search_tool = None
try:
    if os.getenv('SERPER_API_KEY'):
        web_search_tool = SerperDevTool()
        print("Web Search Tool (SerperDev) initialized")
    else:
        print("No SERPER_API_KEY found — agents will use LLM knowledge only")
except Exception as e:
    print(f"Web Search Tool error: {e}")

Web Search Tool (SerperDev) initialized


## Agent Definition

Agents are role-based team members with a **role**, **goal**, **backstory**, and **tools**.

In [5]:
agents_yaml_content = """
content_researcher:
  role: Senior Content Researcher
  goal: Research and gather comprehensive information on given topics to support content creation
  backstory: Experienced researcher with 10+ years in digital marketing, skilled at finding trends and actionable insights.
  tools:
  - web_search
  verbose: true
  allow_delegation: false

content_writer:
  role: Senior Content Writer
  goal: Create engaging, high-quality content based on research and brand guidelines
  backstory: Seasoned content writer who transforms research into compelling marketing narratives.
  tools: []
  verbose: true
  allow_delegation: false

content_editor:
  role: Senior Content Editor
  goal: Review, refine, and ensure content quality meets brand standards and objectives
  backstory: Experienced editor with a keen eye for detail, brand consistency, and audience engagement.
  tools: []
  verbose: true
  allow_delegation: true
"""

agents_config = yaml.safe_load(agents_yaml_content.strip())

agents_config

{'content_researcher': {'role': 'Senior Content Researcher',
  'goal': 'Research and gather comprehensive information on given topics to support content creation',
  'backstory': 'Experienced researcher with 10+ years in digital marketing, skilled at finding trends and actionable insights.',
  'tools': ['web_search'],
  'verbose': True,
  'allow_delegation': False},
 'content_writer': {'role': 'Senior Content Writer',
  'goal': 'Create engaging, high-quality content based on research and brand guidelines',
  'backstory': 'Seasoned content writer who transforms research into compelling marketing narratives.',
  'tools': [],
  'verbose': True,
  'allow_delegation': False},
 'content_editor': {'role': 'Senior Content Editor',
  'goal': 'Review, refine, and ensure content quality meets brand standards and objectives',
  'backstory': 'Experienced editor with a keen eye for detail, brand consistency, and audience engagement.',
  'tools': [],
  'verbose': True,
  'allow_delegation': True}}

In [6]:
tools_map = {'web_search': web_search_tool}

agents = {}
for agent_name, config in agents_config.items():
    agent_tools = [tools_map[t] for t in config.get('tools', []) if t in tools_map and tools_map[t]]

    backstory = config['backstory']
    if config.get('allow_delegation', False):
        backstory += "\n\nIMPORTANT: When delegating tasks, provide parameters as plain strings, not as dictionary objects."

    agents[agent_name] = Agent(
        role=config['role'],
        goal=config['goal'],
        backstory=backstory,
        tools=agent_tools,
        verbose=config.get('verbose', True),
        allow_delegation=config.get('allow_delegation', False),
        llm=llm,
        max_iter=5
    )

researcher = agents['content_researcher']
writer = agents['content_writer']
editor = agents['content_editor']

for name, agent in agents.items():
    print(f"{agent.role}: {len(agent.tools)} tool(s), delegation={'yes' if agent.allow_delegation else 'no'}")

Senior Content Researcher: 1 tool(s), delegation=no
Senior Content Writer: 0 tool(s), delegation=no
Senior Content Editor: 0 tool(s), delegation=yes


## Task Definition

Tasks are structured work items with a **description**, **expected output**, **assigned agent**, and optional **context** (dependencies on other tasks).

In [7]:
tasks_yaml_content = """
research_task:
  description: Research the latest trends and best practices in AI-powered content marketing for 2024. Focus on practical applications, case studies, and emerging technologies.
  agent: content_researcher
  expected_output: A comprehensive research report with key trends, statistics, and actionable insights.
  tools_used:
  - web_search

writing_task:
  description: Create an engaging blog post about AI-powered content marketing based on the research findings. Target audience is marketing professionals.
  agent: content_writer
  expected_output: A well-structured 1200-1500 word blog post with introduction, key sections, examples, and conclusion.
  context:
  - research_task
  tools_used: []

editing_task:
  description: Review and refine the blog post for clarity, engagement, brand consistency, and SEO optimization.
  agent: content_editor
  expected_output: A polished, publication-ready blog post with improved flow, grammar, and SEO.
  context:
  - research_task
  - writing_task
  tools_used: []
"""

tasks_config = yaml.safe_load(tasks_yaml_content.strip())

for task_name, config in tasks_config.items():
    deps = config.get('context', ['None'])
    print(f"{task_name}: agent={config['agent']}, depends_on={', '.join(deps)}")

research_task: agent=content_researcher, depends_on=None
writing_task: agent=content_writer, depends_on=research_task
editing_task: agent=content_editor, depends_on=research_task, writing_task


In [8]:
tasks = {}

for task_name, config in tasks_config.items():
    context = [tasks[dep] for dep in config.get('context', []) if dep in tasks]

    tasks[task_name] = Task(
        description=config['description'],
        agent=agents[config['agent']],
        expected_output=config['expected_output'],
        context=context if context else None
    )

research_task = tasks['research_task']
writing_task = tasks['writing_task']
editing_task = tasks['editing_task']

for i, (name, task) in enumerate(tasks.items(), 1):
    deps = [n for n, t in tasks.items() if task.context and t in (task.context if isinstance(task.context, list) else [task.context])]
    print(f"{i}. {name} -> {task.agent.role} (depends: {', '.join(deps) or 'none'})")

1. research_task -> Senior Content Researcher (depends: none)
2. writing_task -> Senior Content Writer (depends: research_task)
3. editing_task -> Senior Content Editor (depends: research_task, writing_task)


## Crew Orchestration

The **Crew** orchestrates agents and tasks. Process types: **sequential**, **hierarchical**

In [9]:
content_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.sequential,
    verbose=True,
    memory=False,
    max_iter=10
)

print(f"Crew: {len(content_crew.agents)} agents, {len(content_crew.tasks)} tasks, process={content_crew.process.value}")

Crew: 3 agents, 3 tasks, process=sequential


## Execute Crew

In [10]:
#result = content_crew.kickoff()
result = await content_crew.kickoff_async()

print("\nFinal Deliverable:")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: bfb2f9cd-de52-48be-8f94-d6ba8970b680                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the latest trends and best practices in AI-powered content marketing for 2024. Focus on         │
│  practical applications, case studies, and emerging technologies.                                               │
│  ID: b245f968-f9df-40b9-b3f6-e2184eba6d46                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Content Researcher                                                                               │
│                                                                                                                 │
│  Task: Research the latest trends and best practices in AI-powered content marketing for 2024. Focus on         │
│  practical applications, case studies, and emerging technologies.                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'AI-powered content marketing trends 2024'}                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'AI-powered content marketing trends 2024', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': '42 Experts Reveal Top Content Marketing Trends for 2026'...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Content Editor                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Unlocking the Future of AI-Powered Content Marketing in 2024                                               │
│                                                                                                                 │
│  In the fast-evolving landscape of digital marketing, staying ahead of the curve is not just an advantage—it's  │
│  a necessity. As we venture into 2024, AI-powered content marketing is set to revolutionize the way marketers   │
│  engage with their audiences. With its potential to hyper-personalize experiences, optimize content creation,   │
│  and predict consumer behavior, AI is becoming an indispensable tool in the marketer's arsenal. This blog post  │
│  delves into the latest trends, best practices, and emerging technologies in AI-powered content marketing,      │
│  providing insights and strategies for marketing professionals aiming to harness the power of AI.               │
│                                                                                                                 │
│  ### The Rise of AI in Content Marketing: Key Trends                                                            │
│                                                                                                                 │
│  #### 1. Hyper-Personalization Using AI                                                                         │
│                                                                                                                 │
│  In 2024, hyper-personalization is not just a buzzword; it's a critical strategy for brands looking to connect  │
│  with consumers on a deeper level. AI technologies analyze vast amounts of customer data to deliver content     │
│  tailored precisely to individual preferences and behaviors. This level of personalization extends across all   │
│  touchpoints, from email campaigns to social media interactions, ensuring a seamless and engaging user          │
│  experience. By leveraging AI for hyper-personalization, marketers can enhance customer loyalty and drive       │
│  conversions more effectively.                                                                                  │
│                                                                                                                 │
│  #### 2. AI-Powered Content Creation and Optimization                                                           │
│                                                                                                                 │
│  Gone are the days when content creation was entirely manual. AI tools now facilitate the dynamic creation and  │
│  optimization of content, ensuring that every piece is impactful and resonates with the target audience. These  │
│  tools enable companies to generate high-quality content efficiently, adapting it in real-time based on user    │
│  interactions and preferences. For instance, AI can adjust headlines, images, and calls-to-action dynamically,  │
│  ensuring that content remains relevant and engaging.                                                           │
│                                                                                                                 │
│  #### 3. Predictive Analytics                                                                                   │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review and refine the blog post for clarity, engagement, brand consistency, and SEO optimization.        │
│  Agent: Senior Content Editor                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final Deliverable:
### Unlocking the Future of AI-Powered Content Marketing in 2024

In the fast-evolving landscape of digital marketing, staying ahead of the curve is not just an advantage—it's a necessity. As we venture into 2024, AI-powered content marketing is set to revolutionize the way marketers engage with their audiences. With its potential to hyper-personalize experiences, optimize content creation, and predict consumer behavior, AI is becoming an indispensable tool in the marketer's arsenal. This blog post delves into the latest trends, best practices, and emerging technologies in AI-powered content marketing, providing insights and strategies for marketing professionals aiming to harness the power of AI.

### The Rise of AI in Content Marketing: Key Trends

#### 1. Hyper-Personalization Using AI

In 2024, hyper-personalization is not just a buzzword; it's a critical strategy for brands looking to connect with consumers on a deeper level. AI technologies analyze vast amount

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Summary

CrewAI provides role-based multi-agent orchestration with:
- **Agents**: Role-based team members with goals, backstories, and tools
- **Tasks**: Structured work items with dependencies and expected outputs
- **Crew**: Orchestrates agents through sequential, hierarchical, or consensual processes
- **YAML config**: Scalable agent/task definitions without code changes

### Resources
- [CrewAI Documentation](https://docs.crewai.com/)
- [GitHub: crewAI-Inc/crewAI](https://github.com/crewAI-Inc/crewAI)